# Data Preprocessing and Feature Engineering

## Purpose

This notebook prepares the cleaned Phase I diabetes dataset for the machine-learning experiments in Phase II. It begins by validating the input data and documenting the predictor types before any transformations are applied.

The later preprocessing steps will address the representation of binary, ordinal, continuous, and count-based variables, as well as the imbalance in the `Diabetes_binary` target. Any transformation or class-imbalance technique that learns information from the data will be applied only to the training set to prevent data leakage.


## 1. Load the Cleaned Dataset

The cleaned dataset produced during Phase I is loaded as the input for Phase II preprocessing. Initial checks are performed before any new transformations are applied.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/processed/diabetes_cleaned.csv")

df = pd.read_csv(data_path)

print(f"Dataset loaded from: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.head()

Dataset loaded from: ../data/processed/diabetes_cleaned.csv
Rows: 253,680
Columns: 22


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## 2. Initial Data Validation

The structure and quality of the cleaned dataset are checked again before preprocessing. This confirms that the Phase I output has been loaded correctly and that no unexpected missing values or structural changes have been introduced.

In [2]:
validation_summary = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Values": df.isna().sum(),
    "Unique_Values": df.nunique()})

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Total missing values: {df.isna().sum().sum():,}")
print(f"Exact matching rows: {df.duplicated().sum():,}")

display(validation_summary)

Number of rows: 253,680
Number of columns: 22
Total missing values: 0
Exact matching rows: 24,206


,Data_Type,Missing_Values,Unique_Values
HighBP,int64,0,2
HighChol,int64,0,2
CholCheck,int64,0,2
BMI,int64,0,84
Smoker,int64,0,2
Stroke,int64,0,2
HeartDiseaseorAttack,int64,0,2
PhysActivity,int64,0,2
Fruits,int64,0,2
Veggies,int64,0,2


## 3. Target Distribution

The distribution of `Diabetes_binary` is checked because a large difference between the two classes can affect model training and make ordinary accuracy misleading.

In [3]:
target_summary = (
    df["Diabetes_binary"]
    .value_counts()
    .sort_index()
    .rename_axis("Diabetes_binary")
    .reset_index(name="Respondents"))

target_summary["Class"] = target_summary["Diabetes_binary"].map({
    0: "No diabetes",
    1: "Prediabetes/Diabetes"})

target_summary["Percentage"] = (target_summary["Respondents"] / len(df) * 100).round(2)

target_summary = target_summary[["Diabetes_binary", "Class", "Respondents", "Percentage"]]

display(target_summary.style.hide(axis="index"))

Diabetes_binary,Class,Respondents,Percentage
0,No diabetes,218334,86.070000
1,Prediabetes/Diabetes,35346,13.930000


### Interpretation

The cleaned dataset retains the class imbalance identified during Phase I. Most respondents belong to the no-diabetes class (`86.07%`), while the combined prediabetes/diabetes class represents only `13.93%` of the dataset.

This imbalance will be considered during model training and evaluation. Any resampling or class-balancing method will be applied only to the training data so that the test data continues to represent the original population distribution.

## 4. Predictor Types

The predictors are grouped according to how their values should be interpreted. Although every column is stored numerically in the CSV file, the numbers do not all represent the same type of information. Identifying these groups helps prevent inappropriate transformations during preprocessing.

In [4]:
binary_features = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex"]

ordinal_features = ["GenHlth", "Age", "Education", "Income"]

continuous_features = ["BMI"]

count_features = ["MentHlth", "PhysHlth"]

target = "Diabetes_binary"

feature_groups = pd.DataFrame({
    "Feature_Group": [
        "Binary", "Ordinal", "Continuous", "Count-based", "Target"
    ],
    "Features": [
        ", ".join(binary_features),
        ", ".join(ordinal_features),
        ", ".join(continuous_features),
        ", ".join(count_features),
        target
    ],
    "Number_of_Features": [
        len(binary_features),
        len(ordinal_features),
        len(continuous_features),
        len(count_features),
        1
    ]
})

classified_columns = (
    binary_features
    + ordinal_features
    + continuous_features
    + count_features
    + [target])

print(f"Columns classified: {len(classified_columns)}")
print(f"Dataset columns: {df.shape[1]}")
print(f"All columns accounted for: {set(classified_columns) == set(df.columns)}")

display(feature_groups.style.hide(axis="index"))

Columns classified: 22
Dataset columns: 22
All columns accounted for: True


Feature_Group,Features,Number_of_Features
Binary,"HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex",14
Ordinal,"GenHlth, Age, Education, Income",4
Continuous,BMI,1
Count-based,"MentHlth, PhysHlth",2
Target,Diabetes_binary,1
